# AAROH Longitudinal Trajectory Model

This notebook trains the production GRU trajectory classifier using ordered historical observations. Case-level splitting prevents interaction leakage, and padded history is masked so future or padded observations cannot affect the representation. Labels are synthetic demonstration labels, not clinical ground truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/AAROH'
import os
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Expected the AAROH repository at {REPO_DIR}')
os.chdir(REPO_DIR)

In [ ]:
%pip install -q torch scikit-learn
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for Colab trajectory training')

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AAROH')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'trajectory'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path('models/trajectory').mkdir(parents=True, exist_ok=True)
print('Drive checkpoint directory:', CHECKPOINT_DIR)

In [ ]:
!python -m backend.ml.training.train_trajectory \
  --output-dir models/trajectory \
  --checkpoint-dir checkpoints/trajectory \
  --drive-checkpoint-dir $CHECKPOINT_DIR \
  --execution-mode PYTORCH_FROZEN \
  --history-window 10 \
  --batch-size 8 \
  --epochs 5 \
  --fp16

In [ ]:
!python -m backend.ml.training.evaluate_trajectory_model \
  --model-dir models/trajectory \
  --output-file models/trajectory/metrics.json \
  --history-window 10 \
  --case-count 32

In [ ]:
from pathlib import Path
required = [
    Path('models/trajectory/config.json'),
    Path('models/trajectory/metadata.json'),
    Path('models/trajectory/weights'),
    Path('models/trajectory/pytorch_model.bin'),
    Path('models/trajectory/metrics.json'),
    Path('models/trajectory/label_mapping.json'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing exported Trajectory artifacts: {missing}')
print('Longitudinal Trajectory artifacts exported successfully')